In [8]:
# %load_ext autoreload
# %autoreload 2

import sys
sys.path.append('..')

# Import Required Libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
import sentencepiece as spm
from tqdm import tqdm
import math
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from nanochat import GPT, GPTConfig

In [12]:
if __name__ == '__main__':
    config = GPTConfig(  
        vocab_size=1000,  # Small vocab for demo  
        n_layer=4,        # Small model for demo  
        n_head=4,  
        n_kv_head=4,  
        n_embd=256,  
        sequence_len=512  
    )  
    
    model = GPT(config)  
    model.init_weights()  
    
    # Create dummy input tensors  
    batch_size = 2  
    seq_len = 128  
    dummy_ids = torch.randint(0, config.vocab_size, (batch_size, seq_len))  
    dummy_targets = torch.randint(0, config.vocab_size, (batch_size, seq_len))  
    
    # Forward pass with loss  
    print("Running forward pass with loss...")  
    loss = model(dummy_ids, targets=dummy_targets)  
    print(f"Loss: {loss.item():.4f}")  
    
    # Forward pass without loss (inference)  
    print("\nRunning forward pass without loss...")  
    logits_only = model(dummy_ids)  
    print(f"Logits shape: {logits_only.shape}")  
    
    # Generate some tokens (simple greedy decoding)  
    print("\nGenerating tokens...")  
    model.eval()  
    with torch.no_grad():  
        # Start with a dummy prompt  
        prompt = torch.randint(0, config.vocab_size, (1, 10))  
        generated = prompt.clone()  
        
        for _ in range(20):  # Generate 20 tokens  
            logits = model(generated)  
            next_token = logits[:, -1:].argmax(dim=-1)  
            generated = torch.cat([generated, next_token], dim=1)  
        
        print(f"Generated sequence shape: {generated.shape}")  
        print(f"Generated tokens: {generated[0].tolist()}")

Padding vocab_size from 1000 to 1024 for efficiency
Running forward pass with loss...
Loss: 6.9061

Running forward pass without loss...
Logits shape: torch.Size([2, 128, 1000])

Generating tokens...
Generated sequence shape: torch.Size([1, 30])
Generated tokens: [738, 382, 714, 252, 144, 790, 298, 654, 859, 844, 0, 535, 787, 58, 431, 880, 278, 682, 725, 595, 108, 7, 632, 195, 114, 641, 809, 886, 637, 482]
